# ShopMind AI: Semantic Search Demo

This notebook demonstrates semantic product search using sentence embeddings and FAISS.

In [ ]:
import pickle
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import faiss

## 1. Load the Data

In [ ]:
# Load products CSV
df = pd.read_csv('../data/amazon.csv', nrows=5000)
df = df.drop_duplicates(subset=['product_id'], keep='first')
print(f"Loaded {len(df)} unique products")
print(f"\nColumns: {list(df.columns)[:5]}...")
print(f"\nSample product:")
print(df.iloc[0][['product_id', 'product_name', 'about_product', 'rating']])

## 2. Load FAISS Index and ID Map

In [ ]:
# Load FAISS index
index = faiss.read_index('../data/faiss_index.idx')
print(f"FAISS index loaded: {index.ntotal} vectors, dim={index.d}")

# Load product ID mapping
with open('../data/id_map.pkl', 'rb') as f:
    product_ids = pickle.load(f)
print(f"Loaded {len(product_ids)} product IDs")

## 3. Initialize Embedding Model

In [ ]:
model = SentenceTransformer('all-mpnet-base-v2')
print(f"Model loaded: {model.get_sentence_embedding_dimension()}-dim embeddings")

## 4. Semantic Search Function

In [ ]:
def semantic_search(query, k=5):
    """
    Search for products semantically similar to the query.
    
    Args:
        query (str): Natural language search query
        k (int): Number of results to return
    
    Returns:
        DataFrame with top-k products and similarity scores
    """
    # Encode query
    query_embedding = model.encode(query, convert_to_numpy=True).astype('float32')
    query_embedding = normalize([query_embedding])[0]
    
    # Search FAISS index
    distances, indices = index.search(np.array([query_embedding]), k)
    
    # Retrieve results
    results = []
    for i, idx in enumerate(indices[0]):
        product_id = product_ids[idx]
        product = df[df['product_id'] == product_id].iloc[0]
        results.append({
            'product_id': product_id,
            'product_name': product['product_name'],
            'category': product['category'],
            'price': product['discounted_price'],
            'rating': product['rating'],
            'similarity_score': distances[0][i]
        })
    
    return pd.DataFrame(results)

## 5. Example Queries

In [ ]:
# Example 1: Laptop for coding
query1 = "Best laptop for programming and AI development"
print(f"Query: {query1}")
results1 = semantic_search(query1, k=5)
print(results1.to_string())
print("\n" + "="*80 + "\n")

In [ ]:
# Example 2: Budget earphones
query2 = "Affordable wireless headphones with good sound quality"
print(f"Query: {query2}")
results2 = semantic_search(query2, k=5)
print(results2.to_string())
print("\n" + "="*80 + "\n")

In [ ]:
# Example 3: Accessories
query3 = "USB cables and charging adapters"
print(f"Query: {query3}")
results3 = semantic_search(query3, k=5)
print(results3.to_string())

## 6. Embedding Visualization (Optional)

In [ ]:
# Show embedding characteristics
from sentence_transformers import util

queries = [
    "laptop for coding",
    "wireless earphones",
    "USB cable"
]

embeddings = model.encode(queries, convert_to_numpy=True)
embeddings = normalize(embeddings)

# Compute similarity between queries
print("Query Similarity Matrix:")
for i, q1 in enumerate(queries):
    for j, q2 in enumerate(queries):
        if i <= j:
            sim = np.dot(embeddings[i], embeddings[j])
            print(f"{q1:20s} <-> {q2:20s}: {sim:.4f}")